# DW-NOMINATE — what the Voteview data actually is

Voteview publishes three levels, and it is easy to reach for the wrong one:

| file | one row is | size |
| --- | --- | --- |
| `votes` | one member's vote on one roll call | 701 MB |
| `rollcalls` | one roll call — date, question, result | 29.8 MB |
| `members` | **one member in one Congress, as a point in ideology space** | 6.2 MB |

`members` is not a voting record. It is what you get after scaling every
individual vote: each member collapses to a coordinate whose geometry predicts
how they voted. That scaling is the hard part and Voteview has done it, which
is why the 6 MB file is the one worth building on.

This notebook is about reading that coordinate — what it means, and what
reductions of it are worth storing as series.

In [1]:
import sys
sys.path.append("../")     # notebooks/<source>/ helpers
sys.path.append("../../")  # repo root

import csv, collections, statistics
import numpy
from matplotlib import pyplot

from lib import config
import fetch

pyplot.style.use(config.glyfish_style)

members = [r for r in csv.DictReader(open("data/HSall_members.csv"))
           if r["chamber"] in fetch.CHAMBERS and r["nominate_dim1"]]
print(f"{len(members):,} member-Congress rows")

50,728 member-Congress rows


## One row

The scores and the fit diagnostics sit side by side, which is the useful part:
you can see how well the point describes the member rather than taking it on
faith.

In [2]:
row = next(r for r in members if r["bioname"].startswith("PELOSI") and r["congress"] == "100")
for key in ("congress", "chamber", "state_abbrev", "party_code", "bioname",
            "nominate_dim1", "nominate_dim2",
            "nominate_number_of_votes", "nominate_number_of_errors",
            "nominate_geo_mean_probability", "nokken_poole_dim1"):
    print(f"  {key:32s} {row[key]}")


  congress                         100
  chamber                          House
  state_abbrev                     CA
  party_code                       100
  bioname                          PELOSI, Nancy
  nominate_dim1                    -0.489
  nominate_dim2                    -0.174
  nominate_number_of_votes         609
  nominate_number_of_errors        26
  nominate_geo_mean_probability    0.89725
  nokken_poole_dim1                -0.536


`dim1` is the economic left–right axis, roughly −1 to +1. `dim2` is the
secondary axis — historically race and civil rights.

The diagnostics say the estimate is worth trusting: 609 votes went into
placing her, the model mispredicts 26 of them, and it assigns her actual votes
a mean probability of 0.897. Members with too few votes to place are the 224
rows with a blank `nominate_dim1`.

`nokken_poole_dim1` is the alternative scaling. DW-NOMINATE constrains a
member to drift linearly across their whole career; Nokken–Poole re-estimates
them independently each Congress. Use `nominate_dim1` unless you specifically
want within-career movement.

## Every member is a point

Plotting the two coordinates is the clearest statement of what the data is.
Colour by party and polarization stops being an abstraction — it is the
distance between two clouds.

In [3]:
def chamber_rows(congress, chamber="House"):
    return [r for r in members
            if int(r["congress"]) == congress and r["chamber"] == chamber
            and r["nominate_dim2"]]

COLOR = {fetch.DEMOCRAT: "tab:blue", fetch.REPUBLICAN: "tab:red"}

fig, axes = pyplot.subplots(1, 2, figsize=(12, 5.5), sharex=True, sharey=True)
for ax, congress in zip(axes, (88, 119)):
    for code_, label in ((fetch.DEMOCRAT, "Democrat"), (fetch.REPUBLICAN, "Republican")):
        pts = [(float(r["nominate_dim1"]), float(r["nominate_dim2"]))
               for r in chamber_rows(congress) if fetch.party_code(r) == code_]
        ax.scatter(*zip(*pts), s=12, alpha=0.65, color=COLOR[code_], label=label)
    ax.axvline(0, color="grey", lw=0.6, ls=":")
    ax.set_title(f"{congress}th House, {1789 + (congress - 1) * 2}")
    ax.set_xlabel("dim 1 — economic left/right")
axes[0].set_ylabel("dim 2 — the secondary axis")
axes[0].legend(loc="lower left")
fig.suptitle("Every member is a point; polarization is the gap between the clouds")
fig.tight_layout()

Two things to read off that pair.

**The horizontal gap opens.** In 1963 the clouds nearly touch at zero; by 2025
there is clear space between them, and no member sits in it.

**The vertical spread collapses.** In 1963 the Democrats smear upward to +1.0
— that is the Southern/Northern split on civil rights, which is what `dim2`
was measuring. By 2025 the second dimension has largely stopped carrying
information, which is why almost all modern work uses `dim1` alone.

## The distribution splitting

The same fact as a histogram: one mound with a dip in it becomes two mounds
with a gap.

In [4]:
scores = collections.defaultdict(list)
for r in members:
    scores[(int(r["congress"]), r["chamber"], fetch.party_code(r))].append(
        float(r["nominate_dim1"]))

fig, axes = pyplot.subplots(1, 4, figsize=(15, 3.6), sharex=True, sharey=True)
for ax, congress in zip(axes, (88, 100, 110, 119)):
    for code_, colour in ((fetch.DEMOCRAT, "tab:blue"), (fetch.REPUBLICAN, "tab:red")):
        ax.hist(scores.get((congress, "House", code_), []),
                bins=numpy.arange(-1, 1.05, 0.06), alpha=0.7, color=colour)
    ax.set_title(f"{1789 + (congress - 1) * 2}")
    ax.set_xlabel("dim 1")
axes[0].set_ylabel("members")
fig.suptitle("House ideology distribution — the overlap disappears")
fig.tight_layout()

## Two reductions worth storing

A series needs one number per Congress, so something has to collapse those
~450 points. Two measures capture different halves of the same story, and
neither substitutes for the other.

**Distance between party medians** — how far apart the two blocs sit.
**Members inside the other party's range** — how many people are left in
between. The first can rise while the second is already zero, which is exactly
what has happened since 2007.

In [5]:
def median_gap(congress, chamber):
    d = scores.get((congress, chamber, fetch.DEMOCRAT), [])
    r = scores.get((congress, chamber, fetch.REPUBLICAN), [])
    if len(d) < 20 or len(r) < 20:          # before the two-party frame
        return None
    return statistics.median(r) - statistics.median(d)


def moderate_bloc(congress, chamber):
    """Members whose score falls inside the opposing party's range."""
    d = scores.get((congress, chamber, fetch.DEMOCRAT), [])
    r = scores.get((congress, chamber, fetch.REPUBLICAN), [])
    if len(d) < 20 or len(r) < 20:
        return None
    return sum(1 for x in d if x >= min(r)) + sum(1 for x in r if x <= max(d))


congresses = list(range(35, 120))
years = [1789 + (c - 1) * 2 for c in congresses]

fig, (top, bottom) = pyplot.subplots(2, 1, figsize=(11, 7), sharex=True)
for chamber, style in (("House", "-"), ("Senate", "--")):
    top.plot(years, [median_gap(c, chamber) for c in congresses], style, lw=2, label=chamber)
    bottom.plot(years, [moderate_bloc(c, chamber) for c in congresses], style, lw=2, label=chamber)
top.set_ylabel("median gap"); top.set_title("Distance between party medians"); top.legend()
bottom.set_ylabel("members"); bottom.set_xlabel("Year")
bottom.set_title("Members inside the other party's range")
bottom.legend()
fig.tight_layout()

In [6]:
print("  year   House gap  bloc     Senate gap  bloc")
for c in (88, 100, 110, 119):
    print(f"  {1789 + (c - 1) * 2}   "
          f"{median_gap(c, 'House'):.3f}     {moderate_bloc(c, 'House'):3d}       "
          f"{median_gap(c, 'Senate'):.3f}     {moderate_bloc(c, 'Senate'):3d}")


  year   House gap  bloc     Senate gap  bloc
  1963   0.578      88       0.543      31
  1987   0.666      42       0.615       0
  2007   0.805       0       0.709       0
  2025   0.925       0       0.915       0


The bloc hit **zero in the House in 2007** and has stayed there. The median gap
has kept climbing since — 0.805 to 0.925 — so the two measures are not
redundant: one says the parties keep moving apart, the other says there has
been nobody in the middle for two decades either way.

A first cut of stored series is therefore **4**: both measures × House and
Senate, congresses 35–119 (1857–2027). Note the coverage — Democrats and
Republicans do not both exist before the 34th Congress, so neither measure
reaches the 1789 start of the panel.

Anything about *specific votes* — bipartisan-vote fraction, party unity —
needs `rollcalls` and `votes`, which is 730 MB and a different exercise.
See `downloads.ipynb` for the size gate that keeps them out of the default
pull.